In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
from scipy.stats import exponnorm
import arviz as az
import pickle


ModuleNotFoundError: No module named 'iautils'

In [ ]:
def load_cascade_any(path: str):
    # Allow loading for pkl files
    if path.endswith(".pkl"):
        with open(path, "rb") as f:
            cascade_data = pickle.load(f)
        return cascade_data
    else:
        return cascade.load_cascade_from_file(path)

# model to fit
def PulseProfile(x, Speak, sigma, tau, mu):
    """Exponentially modified Gaussian pulse profile"""
    profile = exponnorm.pdf(x, K=tau/sigma, loc=mu, scale=sigma) #Convolution of a gaussian with exponential 
    profile *= Speak / profile.max() 
    return profile

def MCMCpulse(x, mu, data, Speak, noise_rms, draws=1000, tune=500):
    
    with pm.Model() as model:

        # Priors 
        sigma = pm.Uniform('sigma', lower=1e-8, upper=20e-3)   # intrinsic width
        tau   = pm.Uniform('tau',   lower=1e-6, upper=10e-3)   # scattering timescale

        # Model prediction 
        profile = exponnorm.pdf(x, K=tau/sigma, loc=mu, scale=sigma)
        profile_scaled = profile * (Speak / profile.max())

        # Likelihood
        y = pm.Normal('y', mu=profile_scaled, sigma=noise_rms, observed=data)

        idata = pm.sample(draws=draws, tune=tune, random_seed=42, return_inferencedata=True)

    return idata

In [ ]:
filepath = '/Users/meenaseth/gp_analysis/cascade_174258449_norescale_flux_calibrated_recalc_main_beam.pkl'

data = load_cascade_any(filepath)

In [ ]:
# obtain MCMC results
traces = MCMCpulse(xObs, yObs, xErr, yErr, doXerror=True, draws=1000, tune=200)

# compute summary statistics
summary = az.summary(traces)

# best-fit parameters (posterior means)
bf = summary["mean"]

print(bf)